Now, attempt to get solve of a given u, by hard coding the right hand side.
Starting with u(x,y)=1-3x^2+2x^3 again.
But this time we will use the PDE he gave us, 
-div(K grad u) + sigma^2 u = f  strong form aka -ddx(k ddxu) - ddy(k ddy u) + sigma^2 u = f   
int_omega[(k ddx u ddx phi + k ddy u ddy phi) + sigma^2 u phi ] dxdy= int_omega f phi dxdy    weak form
    


In [ ]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Domain is unit square, create initial mesh
N = 64 
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
# Draw(mesh);
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and default=Neumann on the other 3 sides
fes = H1(mesh, order=1, dirichlet="left")
fes.ndof

# Set function that will be used for boundary and RHS
#g = 1 - 3 * x * x + 2 * x * x * x

# Solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(1, BND) # this sets the Dirichlet boundary component to 1, might want to modify later
Draw(gfu)



In [ ]:

# Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()

# # Coefficient function for K(x,y)
Kxy = CoefficientFunction(1+x*x+y*y)
# #Kxy = CoefficientFunction(3)
sigma = 0.5


# Bilinear form
a = BilinearForm(fes)
a += Kxy*grad(u)*grad(phi)*dx+sigma*sigma*u*phi*dx
a.Assemble()

# Right hand side
f = LinearForm(fes)
fakef = 25/4 -12*x + 12*x*y - 12*x*y*y + 69/4 *x*x - 12*x*x*y - 47/2 *x*x*x + 6*y*y
f += fakef*phi*dx
f.Assemble()

# print(a.mat)
# print(f.vec)

In [ ]:
# To see the K(x,y) function, we plot it on our mesh
# Draw(Kxy, mesh)
# Draw the initial state of the grid function 
# Draw(gfu)

# Solve the PDE
gfu.Set(1, BND)
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=f, gf=gfu, pre=c, maxsteps=500, tol=1e-14)
Draw(gfu)

In [ ]:
# WEEK 1 Task
# Test a nice u and find f then work back to it
# what about u(x,y)=1-3x^2+2x^3 ?
plant = CoefficientFunction(1-3*x*x+2*x*x*x)
Draw(plant, mesh)

# Code the right hand side given the u selected
# f = partial w.r.t. x (k(x,y))
# f = - div (k grad u) + sigma^2 u 
# = - d/dx(k grad u) - d/dy(k grad u) + sigma^2 u
# fakef = 25/4 -12*x + 12*x*y-12*x*y**2 + 69/4 *x**2 -12*x**2*y - 47/2 *x**3 + 6*y**2
# Draw(fakef,mesh)
# newf = LinearForm(fes)
# newf += fakef*phi*dx
# newf.Assemble()

# Solve it
# gfu2 = GridFunction(fes)
# gfu2.Set(1, BND)
# c = Preconditioner(a,"local")
# c.Update()
# solvers.BVP(bf=a, lf=newf, gf=gfu2, pre=c)
# Draw(gfu2)